In [ ]:
import json
import os
import re
import time
import csv
import glob
import heapq
from collections import Counter, deque
from difflib import SequenceMatcher

# ==============================================================================
# 1. INTERNAL LOGICAL REPRESENTATION (TERMS, LITERALS, HORN RULES, FOL AST)
# ==============================================================================

class Term:
    """First-Order Logic Term: Constant or Variable (?x)."""
    def __init__(self, name: str, is_var: bool = False):
        self.name = name
        self.is_var = is_var

    def __eq__(self, other):
        return isinstance(other, Term) and self.name == other.name and self.is_var == other.is_var

    def __hash__(self):
        return hash((self.name, self.is_var))

    def __repr__(self):
        return self.name if self.name.startswith('?') else (f"?{self.name}" if self.is_var else self.name)


class Literal:
    """First-Order or Propositional Literal: [¬]Predicate(t1, ..., tk)."""
    def __init__(self, pred: str, args=(), negated: bool = False):
        self.pred = pred
        self.args = tuple(
            Term(a, is_var=(a.startswith('?') or (len(a) == 1 and a in 'XYZUVWxyzuvw')))
            if isinstance(a, str) else a
            for a in args
        )
        self.negated = negated

    def negate(self):
        return Literal(self.pred, self.args, not self.negated)

    def is_ground(self):
        return all(not a.is_var for a in self.args)

    def __eq__(self, other):
        return (isinstance(other, Literal) and self.pred == other.pred and
                self.args == other.args and self.negated == other.negated)

    def __hash__(self):
        return hash((self.pred, self.args, self.negated))

    def __repr__(self):
        prefix = "¬" if self.negated else ""
        if not self.args:
            return f"{prefix}{self.pred}"
        return f"{prefix}{self.pred}({', '.join(repr(a) for a in self.args)})"


class Rule:
    """Definite / Horn Clause: body_1 ∧ ... ∧ body_m -> head."""
    def __init__(self, body, head: Literal, raw_text: str = ""):
        self.body = tuple(body)
        self.head = head
        self.raw_text = raw_text

    def __repr__(self):
        b_str = " ∧ ".join(repr(b) for b in self.body)
        return f"({b_str} → {repr(self.head)})"


class Expr:
    """Base class for full First-Order Logic AST expressions."""
    pass

class Atom(Expr):
    def __init__(self, pred, args):
        self.pred = pred
        self.args = tuple(args)
    def __repr__(self):
        return f"{self.pred}({', '.join(self.args)})"

class Not(Expr):
    def __init__(self, sub):
        self.sub = sub
    def __repr__(self):
        return f"¬({self.sub})"

class BinOp(Expr):
    def __init__(self, op, left, right):
        self.op = op
        self.left = left
        self.right = right
    def __repr__(self):
        return f"({self.left} {self.op} {self.right})"

class Quant(Expr):
    def __init__(self, q, var, sub):
        self.q = q
        self.var = var
        self.sub = sub
    def __repr__(self):
        return f"{self.q}{self.var}({self.sub})"

In [ ]:
# ==============================================================================
# 2. CLASSICAL UNIFICATION, FORWARD CHAINING, AND BACKWARD CHAINING (DFS/BFS/UCS/A*)
# ==============================================================================

def substitute_literal(lit: Literal, theta: dict) -> Literal:
    new_args = []
    for a in lit.args:
        cur = a
        while isinstance(cur, Term) and cur.is_var and cur in theta:
            cur = theta[cur]
        new_args.append(cur)
    return Literal(lit.pred, new_args, lit.negated)


def unify_terms(t1: Term, t2: Term, theta: dict):
    if theta is None:
        return None
    while t1.is_var and t1 in theta:
        t1 = theta[t1]
    while t2.is_var and t2 in theta:
        t2 = theta[t2]
    if t1 == t2:
        return theta
    if t1.is_var:
        new_theta = dict(theta)
        new_theta[t1] = t2
        return new_theta
    if t2.is_var:
        new_theta = dict(theta)
        new_theta[t2] = t1
        return new_theta
    return None


def unify_literals(l1: Literal, l2: Literal, theta=None):
    if theta is None:
        theta = {}
    if l1.pred != l2.pred or l1.negated != l2.negated or len(l1.args) != len(l2.args):
        return None
    for a1, a2 in zip(l1.args, l2.args):
        theta = unify_terms(a1, a2, theta)
        if theta is None:
            return None
    return theta


def forward_chaining_fol(facts, rules, query: Literal, max_iters: int = 50):
    """Data-driven Forward Chaining with First-Order Unification."""
    known = set(facts)
    trace = [f"Initial Facts ({len(known)}): " + ", ".join(repr(f) for f in sorted(known, key=repr)[:6])]
    if query in known:
        trace.append(f"Goal {repr(query)} is directly in Initial Facts.")
        return True, known, trace, 0

    def match_body(body_lits, idx, theta):
        if idx == len(body_lits):
            yield theta
            return
        target = substitute_literal(body_lits[idx], theta)
        for kf in list(known):
            new_theta = unify_literals(target, kf, theta)
            if new_theta is not None:
                yield from match_body(body_lits, idx + 1, new_theta)

    iteration = 0
    rule_firings = 0
    changed = True
    while changed and iteration < max_iters:
        changed = False
        iteration += 1
        for rule in rules:
            for theta in match_body(rule.body, 0, {}):
                derived = substitute_literal(rule.head, theta)
                if derived not in known and derived.is_ground():
                    known.add(derived)
                    changed = True
                    rule_firings += 1
                    prem_str = " ∧ ".join(repr(substitute_literal(b, theta)) for b in rule.body)
                    trace.append(f"Step {rule_firings} [FC]: {prem_str} ⇒ {repr(derived)} (via {rule})")
                    if derived == query:
                        trace.append(f"Goal {repr(query)} proved via Forward Chaining in {rule_firings} steps!")
                        return True, known, trace, rule_firings

    trace.append(f"Forward Chaining reached fixed point ({iteration} iters, {rule_firings} firings); {repr(query)} not derived.")
    return (query in known), known, trace, rule_firings


def backward_chaining_search(facts, rules, query: Literal, strategy: str = "astar", max_expansions: int = 600):
    """
    Goal-driven Backward Chaining supporting DFS, BFS, UCS, and A* Heuristic Search
    with First-Order Unification and cycle detection.
    """
    known = set(facts)
    if query in known:
        return True, [f"Goal {repr(query)} is a known fact."], 1

    rules_by_pred = {}
    for r in rules:
        rules_by_pred.setdefault((r.head.pred, r.head.negated), []).append(r)

    def heuristic(goals):
        h = 0
        for g in goals:
            if g in known:
                continue
            elif (g.pred, g.negated) in rules_by_pred:
                h += 1
            else:
                h += 3
        return h

    init_goals = (query,)
    counter = 0
    if strategy in ("astar", "ucs"):
        f0 = heuristic(init_goals) if strategy == "astar" else 0
        frontier = [(f0, 0, counter, init_goals, [f"Init Goal: {repr(query)}"])]
    else:
        frontier = deque([(init_goals, 0, [f"Init Goal: {repr(query)}"])])

    visited = set()
    expansions = 0

    while frontier and expansions < max_expansions:
        expansions += 1
        if strategy in ("astar", "ucs"):
            f_val, g_val, _, goals, trace = heapq.heappop(frontier)
        elif strategy == "bfs":
            goals, g_val, trace = frontier.popleft()
        else:
            goals, g_val, trace = frontier.pop()

        if not goals:
            trace.append(f"All subgoals discharged ({strategy.upper()}, {expansions} expansions)!")
            return True, trace, expansions

        if goals in visited:
            continue
        visited.add(goals)

        first_goal = goals[0]
        rest_goals = goals[1:]

        for kf in known:
            theta = unify_literals(first_goal, kf, {})
            if theta is not None:
                new_goals = tuple(substitute_literal(g, theta) for g in rest_goals)
                new_trace = trace + [f"Fact Match: {repr(first_goal)} = {repr(kf)}"]
                if not new_goals:
                    new_trace.append(f"All subgoals discharged ({strategy.upper()}, {expansions} expansions)!")
                    return True, new_trace, expansions
                if new_goals not in visited:
                    counter += 1
                    new_g = g_val + 1
                    if strategy in ("astar", "ucs"):
                        h_val = heuristic(new_goals) if strategy == "astar" else 0
                        heapq.heappush(frontier, (new_g + h_val, new_g, counter, new_goals, new_trace))
                    else:
                        frontier.append((new_goals, new_g, new_trace))

        cand_rules = rules_by_pred.get((first_goal.pred, first_goal.negated), [])
        for r_idx, rule in enumerate(cand_rules):
            std_map = {}
            for lit in (rule.head,) + rule.body:
                for a in lit.args:
                    if a.is_var and a not in std_map:
                        std_map[a] = Term(f"{a.name}_{expansions}_{r_idx}", is_var=True)
            r_head = substitute_literal(rule.head, std_map)
            r_body = [substitute_literal(b, std_map) for b in rule.body]

            theta = unify_literals(r_head, first_goal, {})
            if theta is not None:
                sub_body = [substitute_literal(b, theta) for b in r_body]
                if first_goal in sub_body:
                    continue
                sub_rest = [substitute_literal(g, theta) for g in rest_goals]
                new_goals = tuple(sub_body + sub_rest)
                if new_goals not in visited:
                    counter += 1
                    new_g = g_val + len(r_body)
                    new_trace = trace + [f"Backchain {repr(first_goal)} via {rule} → {sub_body}"]
                    if strategy in ("astar", "ucs"):
                        h_val = heuristic(new_goals) if strategy == "astar" else 0
                        heapq.heappush(frontier, (new_g + h_val, new_g, counter, new_goals, new_trace))
                    else:
                        frontier.append((new_goals, new_g, new_trace))

    return False, [f"Backward Chaining ({strategy.upper()}) exhausted search ({expansions} expansions); {repr(query)} not entailed."], expansions

In [ ]:
# ==============================================================================
# 3. FIRST-ORDER LOGIC RESOLUTION, SKOLEMIZATION, TSEITIN CNF & DPLL ENGINE
# ==============================================================================

def balance_parens(s: str) -> str:
    depth = 0
    min_depth = 0
    for ch in s:
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
            if depth < min_depth:
                min_depth = depth
    if min_depth < 0:
        m = re.match(r'^((?:[∀∃]\s*[a-zA-Z0-9_]+\s*)*)(.*)$', s)
        prefix, rest = m.group(1), m.group(2)
        s = prefix + ('(' * (-min_depth)) + rest
    depth = 0
    for ch in s:
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
    if depth > 0:
        s = s + (')' * depth)
    return s


def tokenize_fol(s: str):
    s = balance_parens(s)
    s = s.replace('<->', '↔').replace('->', '→').replace('&', '∧').replace('|', '∨').replace('~', '¬')
    token_spec = [
        ('QUANT', r'[∀∃]\s*[a-zA-Z0-9_]+'),
        ('OP', r'[∧∨¬→↔⊕=]'),
        ('LPAREN', r'\('),
        ('RPAREN', r'\)'),
        ('COMMA', r','),
        ('ID', r'[a-zA-Z0-9_\-\.]+'),
        ('SKIP', r'\s+'),
    ]
    tok_regex = '|'.join('(?P<%s>%s)' % pair for pair in token_spec)
    tokens = []
    for mo in re.finditer(tok_regex, s):
        kind = mo.lastgroup
        val = mo.group()
        if kind == 'SKIP':
            continue
        if kind == 'QUANT':
            tokens.append(('QUANT', (val[0], val[1:].strip())))
        else:
            tokens.append((kind, val))
    return tokens


def parse_fol_formula(s: str) -> Expr:
    tokens = tokenize_fol(s)
    pos = [0]
    def peek():
        return tokens[pos[0]] if pos[0] < len(tokens) else (None, None)
    def consume():
        t = peek()
        pos[0] += 1
        return t

    def parse_iff():
        left = parse_imp()
        while peek()[1] in ('↔', '→'):
            op = consume()[1]
            right = parse_imp()
            left = BinOp(op, left, right)
        return left

    def parse_imp():
        left = parse_xor()
        while peek()[1] == '⊕':
            op = consume()[1]
            right = parse_xor()
            left = BinOp(op, left, right)
        return left

    def parse_xor():
        left = parse_or()
        while peek()[1] == '∨':
            op = consume()[1]
            right = parse_or()
            left = BinOp(op, left, right)
        return left

    def parse_or():
        left = parse_and()
        while peek()[1] == '∧':
            op = consume()[1]
            right = parse_and()
            left = BinOp(op, left, right)
        return left

    def parse_and():
        k, v = peek()
        if k == 'OP' and v == '¬':
            consume()
            return Not(parse_and())
        if k == 'QUANT':
            consume()
            q, var = v
            return Quant(q, var, parse_iff())
        if k == 'LPAREN':
            consume()
            expr = parse_iff()
            if peek()[0] == 'RPAREN':
                consume()
            return expr
        if k == 'ID':
            name = consume()[1]
            if peek()[0] == 'LPAREN':
                consume()
                args = []
                while peek()[0] not in ('RPAREN', None):
                    if peek()[0] == 'COMMA':
                        consume()
                    elif peek()[0] == 'ID':
                        args.append(consume()[1])
                    else:
                        consume()
                if peek()[0] == 'RPAREN':
                    consume()
                return Atom(name, args)
            elif peek()[1] == '=':
                consume()
                other = consume()[1] if peek()[0] == 'ID' else 'c'
                return Atom('Equals', [name, other])
            else:
                return Atom(name, [])
        consume()
        return Atom('TrueAtom', [])

    exprs = []
    while pos[0] < len(tokens):
        if peek()[0] == 'RPAREN':
            consume()
            continue
        exprs.append(parse_iff())
    if not exprs:
        return Atom('TrueAtom', [])
    res = exprs[0]
    for e in exprs[1:]:
        res = BinOp('∧', res, e)
    return res


def collect_info(expr, bound=None, consts=None, preds=None, free_vars=None):
    if bound is None: bound = set()
    if consts is None: consts = set()
    if preds is None: preds = set()
    if free_vars is None: free_vars = set()
    if isinstance(expr, Atom):
        if expr.pred not in ('TrueAtom', 'Equals'):
            preds.add((expr.pred, len(expr.args)))
        for a in expr.args:
            if a in bound:
                pass
            elif len(a) == 1 and a in 'xyzuvw':
                free_vars.add(a)
            else:
                consts.add(a)
    elif isinstance(expr, Not):
        collect_info(expr.sub, bound, consts, preds, free_vars)
    elif isinstance(expr, BinOp):
        collect_info(expr.left, bound, consts, preds, free_vars)
        collect_info(expr.right, bound, consts, preds, free_vars)
    elif isinstance(expr, Quant):
        collect_info(expr.sub, bound | {expr.var}, consts, preds, free_vars)
    return consts, preds, free_vars


def collect_subj_consts(expr, bound=None, subjs=None):
    if bound is None: bound = set()
    if subjs is None: subjs = Counter()
    if isinstance(expr, Atom):
        if expr.args:
            a0 = expr.args[0]
            if a0 not in bound and not (len(a0) == 1 and a0 in 'xyzuvw'):
                subjs[a0] += 1
    elif isinstance(expr, Not):
        collect_subj_consts(expr.sub, bound, subjs)
    elif isinstance(expr, BinOp):
        collect_subj_consts(expr.left, bound, subjs)
        collect_subj_consts(expr.right, bound, subjs)
    elif isinstance(expr, Quant):
        collect_subj_consts(expr.sub, bound | {expr.var}, subjs)
    return subjs


def subst_fol(expr, mapping, pred_map=None):
    if isinstance(expr, Atom):
        p = pred_map.get(expr.pred, expr.pred) if pred_map else expr.pred
        return Atom(p, [mapping.get(a, a) for a in expr.args])
    elif isinstance(expr, Not):
        return Not(subst_fol(expr.sub, mapping, pred_map))
    elif isinstance(expr, BinOp):
        return BinOp(expr.op, subst_fol(expr.left, mapping, pred_map), subst_fol(expr.right, mapping, pred_map))
    elif isinstance(expr, Quant):
        m2 = {k: v for k, v in mapping.items() if k != expr.var}
        return Quant(expr.q, expr.var, subst_fol(expr.sub, m2, pred_map))
    return expr


def ground_expr(expr, universe):
    if isinstance(expr, Atom):
        return expr
    elif isinstance(expr, Not):
        return Not(ground_expr(expr.sub, universe))
    elif isinstance(expr, BinOp):
        return BinOp(expr.op, ground_expr(expr.left, universe), ground_expr(expr.right, universe))
    elif isinstance(expr, Quant):
        sub_exprs = [ground_expr(subst_fol(expr.sub, {expr.var: c}), universe) for c in universe]
        op = '∧' if expr.q == '∀' else '∨'
        res = sub_exprs[0]
        for se in sub_exprs[1:]:
            res = BinOp(op, res, se)
        return res


def to_cnf_tseitin(expr):
    atom_map = {}
    clauses = []
    next_var = [1]
    primary_vars = set()
    def get_var(name):
        if name not in atom_map:
            v = next_var[0]
            next_var[0] += 1
            atom_map[name] = v
            primary_vars.add(v)
        return atom_map[name]
    def new_aux():
        v = next_var[0]
        next_var[0] += 1
        return v

    v_true = new_aux()
    clauses.append([v_true])

    def encode(e):
        if isinstance(e, Atom):
            if e.pred == 'TrueAtom':
                return v_true
            if e.pred == 'Equals':
                if len(e.args) == 2 and e.args[0] == e.args[1]:
                    return v_true
                else:
                    return -v_true
            return get_var(repr(e))
        elif isinstance(e, Not):
            return -encode(e.sub)
        elif isinstance(e, BinOp):
            l = encode(e.left)
            r = encode(e.right)
            aux = new_aux()
            if e.op == '∧':
                clauses.append([-aux, l])
                clauses.append([-aux, r])
                clauses.append([aux, -l, -r])
            elif e.op == '∨':
                clauses.append([aux, -l])
                clauses.append([aux, -r])
                clauses.append([-aux, l, r])
            elif e.op == '→':
                clauses.append([aux, l])
                clauses.append([aux, -r])
                clauses.append([-aux, -l, r])
            elif e.op == '↔':
                clauses.append([-aux, -l, r])
                clauses.append([-aux, l, -r])
                clauses.append([aux, l, r])
                clauses.append([aux, -l, -r])
            elif e.op == '⊕':
                clauses.append([-aux, l, r])
                clauses.append([-aux, -l, -r])
                clauses.append([aux, -l, r])
                clauses.append([aux, l, -r])
            return aux
    root = encode(expr)
    clauses.append([root])
    return clauses, primary_vars


def dpll_solve(clauses, primary_vars, max_steps=1500):
    steps = [0]
    def solve(cls):
        steps[0] += 1
        if steps[0] > max_steps:
            return True
        while True:
            unit_lits = []
            for c in cls:
                if len(c) == 0:
                    return False
                if len(c) == 1:
                    unit_lits.append(c[0])
            if not unit_lits:
                break
            unit_set = set(unit_lits)
            for l in unit_set:
                if -l in unit_set:
                    return False
            new_cls = []
            for c in cls:
                if any(l in unit_set for l in c):
                    continue
                nc = [x for x in c if -x not in unit_set]
                if len(nc) == 0:
                    return False
                new_cls.append(nc)
            cls = new_cls
        if not cls:
            return True
        cls.sort(key=len)
        chosen = None
        for c in cls:
            for lit in c:
                if abs(lit) in primary_vars:
                    chosen = lit
                    break
            if chosen is not None:
                break
        if chosen is None:
            chosen = cls[0][0]
        return solve(cls + [[chosen]]) or solve(cls + [[-chosen]])
    return solve(clauses)


def is_typo_match(s1: str, s2: str) -> bool:
    a, b = s1.lower(), s2.lower()
    if a == b:
        return True
    if (a.startswith(b) or b.startswith(a)) and abs(len(a) - len(b)) <= 6:
        return True
    if SequenceMatcher(None, a, b).ratio() >= 0.88 and abs(len(a) - len(b)) <= 3:
        return True
    return False


def split_camel(s: str):
    s = re.sub(r'([a-z0-9])([A-Z])', r'\1 \2', s)
    tokens = [t.lower() for t in re.split(r'[^a-zA-Z0-9]+', s) if t]
    stop = {'is', 'are', 'was', 'were', 'be', 'been', 'to', 'of', 'in', 'on', 'at', 'by', 'for', 'with', 'that', 'the', 'a', 'an', 'more'}
    return [t for t in tokens if t not in stop]


def normalize_atom_signature(atom: Atom):
    pred_toks = split_camel(atom.pred)
    neg = False
    clean_toks = []
    for t in pred_toks:
        if t in ('not', 'no', 'never', 'unaware', 'inactive', 'disinterested', 'unemployed'):
            if t == 'not':
                neg = not neg
            elif t.startswith('un') and len(t) > 4:
                neg = not neg
                clean_toks.append(t[2:])
            else:
                clean_toks.append(t)
        elif t in ('do', 'does', 'did'):
            continue
        else:
            clean_toks.append(t)
    arg_toks = []
    for a in atom.args[1:]:
        if len(a) == 1 and a in 'xyzuvw':
            arg_toks.append(a)
        else:
            arg_toks.extend(split_camel(a))
    all_toks = []
    for t in clean_toks + arg_toks:
        if len(t) > 4 and t.endswith('s') and not t.endswith('ss'):
            t = t[:-1]
        all_toks.append(t)
    subj = atom.args[0] if atom.args else 'c0'
    return neg, subj, tuple(sorted(set(all_toks)))


def align_compound_atoms(exprs):
    atoms = []
    def gather(e):
        if isinstance(e, Atom):
            if e.pred not in ('TrueAtom', 'Equals'):
                atoms.append(e)
        elif isinstance(e, Not): gather(e.sub)
        elif isinstance(e, BinOp): gather(e.left); gather(e.right)
        elif isinstance(e, Quant): gather(e.sub)
    for e in exprs:
        gather(e)

    sigs = {}
    for a in atoms:
        key = (a.pred, a.args[1:])
        if key not in sigs:
            neg, _, toks = normalize_atom_signature(a)
            sigs[key] = (neg, toks)

    key_list = list(sigs.keys())
    replacement = {}
    for i, k1 in enumerate(key_list):
        neg1, toks1 = sigs[k1]
        if not toks1:
            continue
        for k2 in key_list[:i]:
            neg2, toks2 = sigs[k2]
            if not toks2:
                continue
            set1, set2 = set(toks1), set(toks2)
            inter = len(set1 & set2)
            union = len(set1 | set2)
            if union > 0 and (inter / union >= 0.75 or (inter >= 2 and inter == min(len(set1), len(set2)) and max(len(set1), len(set2)) - inter <= 1)):
                canon = replacement.get(k2, (False, k2[0], k2[1]))
                flip = (neg1 != neg2) ^ canon[0]
                replacement[k1] = (flip, canon[1], canon[2])
                break

    def rewrite(e):
        if isinstance(e, Atom):
            if e.pred in ('TrueAtom', 'Equals') or not e.args:
                return e
            key = (e.pred, e.args[1:])
            if key in replacement:
                flip, cpred, cextra = replacement[key]
                new_atom = Atom(cpred, (e.args[0],) + cextra)
                return Not(new_atom) if flip else new_atom
            return e
        elif isinstance(e, Not):
            return Not(rewrite(e.sub))
        elif isinstance(e, BinOp):
            return BinOp(e.op, rewrite(e.left), rewrite(e.right))
        elif isinstance(e, Quant):
            return Quant(e.q, e.var, rewrite(e.sub))
        return e

    return [rewrite(e) for e in exprs]


def solve_fol_exprs(p_exprs, c_expr, has_existential=False, use_compound_align=False):
    all_consts = set()
    all_preds = set()
    subj_counts = Counter()
    for pe in p_exprs + [c_expr]:
        cs, ps, _ = collect_info(pe)
        all_consts |= cs
        all_preds |= ps
        collect_subj_consts(pe, subjs=subj_counts)

    const_list = sorted(all_consts, key=len, reverse=True)
    const_map = {}
    for i, c1 in enumerate(const_list):
        for c2 in const_list[:i]:
            if is_typo_match(c1, c2):
                const_map[c1] = const_map.get(c2, c2)
                break
    pred_list = sorted({p for p, _ in all_preds}, key=len, reverse=True)
    pred_map = {}
    for i, p1 in enumerate(pred_list):
        for p2 in pred_list[:i]:
            if is_typo_match(p1, p2):
                pred_map[p1] = pred_map.get(p2, p2)
                break

    for pe in p_exprs:
        if isinstance(pe, Atom) and pe.pred == 'Equals' and len(pe.args) == 2:
            const_map[pe.args[0]] = const_map.get(pe.args[1], pe.args[1])

    c_subjs = collect_subj_consts(c_expr)
    if c_subjs:
        default_c = c_subjs.most_common(1)[0][0]
    elif subj_counts:
        default_c = subj_counts.most_common(1)[0][0]
    else:
        default_c = sorted(all_consts)[0] if all_consts else 'c0'
    default_c = const_map.get(default_c, default_c)

    fixed_exprs = []
    for pe in p_exprs + [c_expr]:
        pe = subst_fol(pe, const_map, pred_map)
        cs, _, fvs = collect_info(pe)
        if fvs:
            pe_subjs = collect_subj_consts(pe)
            if pe_subjs and use_compound_align:
                main_c = pe_subjs.most_common(1)[0][0]
                pe = subst_fol(pe, {fv: main_c for fv in fvs})
            elif cs:
                main_c = sorted(cs)[0] if not use_compound_align else default_c
                pe = subst_fol(pe, {fv: main_c for fv in fvs})
            else:
                pe = subst_fol(pe, {fv: default_c for fv in fvs})
        fixed_exprs.append(pe)

    if use_compound_align:
        fixed_exprs = align_compound_atoms(fixed_exprs)

    p_exprs_f = fixed_exprs[:-1]
    c_expr_f = fixed_exprs[-1]

    c_consts, _, _ = collect_info(c_expr_f)
    p_consts = set()
    for pe in p_exprs_f:
        cs, _, _ = collect_info(pe)
        p_consts |= cs

    if use_compound_align:
        universe = []
        for c, _ in subj_counts.most_common():
            mc = const_map.get(c, c)
            if mc not in universe and len(universe) < 4:
                universe.append(mc)
        for c in sorted(c_consts):
            if c not in universe and len(universe) < 4:
                universe.append(c)
    else:
        universe = sorted(c_consts)
        for c in sorted(p_consts):
            if c not in universe and len(universe) < 4:
                universe.append(c)

    if len(universe) < 2:
        universe.append('_sk0')
    if len(universe) < 3 and has_existential:
        universe.append('_sk1')

    kb = p_exprs_f[0]
    for pe in p_exprs_f[1:]:
        kb = BinOp('∧', kb, pe)
    g_kb = ground_expr(kb, universe)
    g_c = ground_expr(c_expr_f, universe)

    cls_neg, pvars_neg = to_cnf_tseitin(BinOp('∧', g_kb, Not(g_c)))
    entailed = not dpll_solve(cls_neg, pvars_neg)

    cls_pos, pvars_pos = to_cnf_tseitin(BinOp('∧', g_kb, g_c))
    contradicted = not dpll_solve(cls_pos, pvars_pos)

    if entailed and not contradicted:
        return 'True'
    elif contradicted and not entailed:
        return 'False'
    else:
        return 'Uncertain'

In [ ]:
# ==============================================================================
# 4. NATURAL LANGUAGE PARSERS (RULETAKER, GENERAL NL->FOL, COUNTERBENCH SCM)
# ==============================================================================

IRREGULAR_VERBS = {
    'sold': 'sell', 'sells': 'sell', 'selling': 'sell',
    'made': 'make', 'makes': 'make', 'making': 'make',
    'created': 'create', 'creates': 'create',
    'built': 'build', 'builds': 'build',
    'wrote': 'write', 'written': 'write', 'writes': 'write',
    'went': 'go', 'goes': 'go', 'going': 'go',
    'had': 'have', 'has': 'have', 'having': 'have',
    'did': 'do', 'does': 'do',
    'sat': 'sit', 'sits': 'sit',
    'took': 'take', 'takes': 'take', 'taken': 'take',
    'knew': 'know', 'knows': 'know', 'known': 'know',
    'children': 'child', 'people': 'person', 'men': 'man', 'women': 'woman',
    'mice': 'mouse', 'teeth': 'tooth', 'feet': 'foot',
    'studies': 'study', 'studied': 'study',
    'carries': 'carry', 'carried': 'carry',
    'watches': 'watch', 'watched': 'watch',
    'teaches': 'teach', 'taught': 'teach',
    'wishes': 'wish', 'wished': 'wish',
    'fishes': 'fish',
}

STOPWORDS = {
    'a', 'an', 'the', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'will', 'would', 'can', 'could', 'may', 'might', 'must', 'shall', 'should',
    'do', 'does', 'did', 'to', 'of', 'in', 'on', 'at', 'by', 'for', 'with',
    'from', 'into', 'onto', 'that', 'which', 'who', 'whom', 'whose', 'this',
    'these', 'those', 'their', 'his', 'her', 'its', 'they', 'them', 'he', 'she',
    'it', 'someone', 'something', 'anyone', 'anything', 'everyone', 'everything',
    'person', 'thing', 'individual', 'one', 'ones', 'very', 'often', 'regularly',
    'always', 'usually', 'sometimes', 'also', 'both', 'today', 'now'
}

def stem_word(w: str) -> str:
    w = w.lower().strip(".,!?;:'()[]")
    if not w:
        return ""
    if w in IRREGULAR_VERBS:
        return IRREGULAR_VERBS[w]
    if len(w) > 5 and w.endswith('ing'):
        base = w[:-3]
        return base + 'e' if base.endswith(('at', 'iz', 'ut', 'iv')) else base
    if len(w) > 4 and w.endswith('ed'):
        if w.endswith('ied'):
            return w[:-3] + 'y'
        base = w[:-2]
        return base + 'e' if base.endswith(('at', 'iz', 'ut', 'iv', 'am', 'ir', 'ur')) else base
    if len(w) > 4 and w.endswith('es'):
        if w.endswith('ies'):
            return w[:-3] + 'y'
        if w.endswith(('ches', 'shes', 'xes', 'zes', 'sses')):
            return w[:-2]
        return w[:-1]
    if len(w) > 3 and w.endswith('s') and not w.endswith(('ss', 'us', 'is')):
        return w[:-1]
    return w


def phrase_to_pred(phrase: str, arg: str = "x") -> Expr:
    p = phrase.strip().rstrip('.').strip()
    p_low = p.lower()
    m_nei = re.match(r'^(?:is\s+|are\s+|was\s+|were\s+)?neither\s+(.*?)\s+nor\s+(.*)$', p_low)
    if m_nei:
        return BinOp('∧', Not(phrase_to_pred(m_nei.group(1), arg)), Not(phrase_to_pred(m_nei.group(2), arg)))
    m_eith = re.match(r'^(?:is\s+|are\s+|was\s+|were\s+)?either\s+(.*?)\s+or\s+(.*?)(?:,\s*or\s+both)?$', p_low)
    if m_eith:
        op = '∨' if 'or both' in p_low else '⊕'
        return BinOp(op, phrase_to_pred(m_eith.group(1), arg), phrase_to_pred(m_eith.group(2), arg))

    neg = False
    for neg_pat in [r'\bnot\b', r'\bnever\b', r'\bno\b', r"don't\b", r"doesn't\b", r"didn't\b", r"cannot\b", r"can't\b", r"won't\b"]:
        if re.search(neg_pat, p_low):
            neg = not neg
            p_low = re.sub(neg_pat, ' ', p_low)

    words = [re.sub(r"[^a-z0-9]", "", w) for w in p_low.split()]
    norm_words = []
    for w in words:
        if not w:
            continue
        if w in ('unaware', 'inactive', 'disinterested', 'independent', 'unemployed', 'unmarried', 'illegal', 'impossible'):
            neg = not neg
            if w.startswith(('un', 'in', 'il', 'im')):
                w = w[2:]
            elif w.startswith('dis'):
                w = w[3:]
        st = stem_word(w)
        if st and st not in STOPWORDS:
            norm_words.append(st)
    if not norm_words:
        norm_words = ['thing']
    atom = Atom("_".join(sorted(set(norm_words))), [arg])
    return Not(atom) if neg else atom


def parse_nl_sentence_to_fol(sentence: str, known_entities: set) -> Expr:
    s = sentence.strip().rstrip('.').strip()
    if not s:
        return Atom('TrueAtom', [])
    s_low = s.lower()

    m_types = re.match(r'^there\s+are\s+[a-z0-9]+\s+(?:types|kinds|categories)\s+of\s+(.*?):\s*(.*)$', s_low)
    if m_types:
        cat = m_types.group(1).strip()
        items = [x.strip() for x in re.split(r',|\band\b', m_types.group(2)) if x.strip()]
        if items:
            disj = phrase_to_pred(items[0], 'x')
            for it in items[1:]:
                disj = BinOp('∨', disj, phrase_to_pred(it, 'x'))
            return Quant('∀', 'x', BinOp('→', phrase_to_pred(cat, 'x'), disj))

    if ' if and only if ' in s_low:
        parts = re.split(r'\bif and only if\b', s, flags=re.IGNORECASE)
        if len(parts) == 2:
            return BinOp('↔', parse_nl_sentence_to_fol(parts[0], known_entities),
                         parse_nl_sentence_to_fol(parts[1], known_entities))

    m_if = re.match(r'^(?:if|when|assuming|given that|based on)\s+(.*?)(?:,\s*then\s+|,\s*we can (?:say|conclude)\s+|,\s*it (?:follows|is evident) that\s+|,\s+)(.*)$', s, flags=re.IGNORECASE)
    if m_if:
        ant_str, con_str = m_if.group(1).strip(), m_if.group(2).strip()
        has_ent = any(ent in s_low for ent in known_entities)
        if not has_ent or re.search(r'\b(someone|something|anyone|anything|an?\s+[a-z]+)\b', ant_str.lower()):
            return Quant('∀', 'x', BinOp('→', phrase_to_pred(ant_str, 'x'), phrase_to_pred(con_str, 'x')))
        else:
            return BinOp('→', parse_nl_sentence_to_fol(ant_str, known_entities),
                         parse_nl_sentence_to_fol(con_str, known_entities))

    m_no = re.match(r'^(?:no\s+one|nobody|nothing)\s+(?:who|that|which)\s+(.*?)\s+(?:is|are|was|were|will|can|does|do)\s+(.*)$', s_low)
    if m_no:
        return Quant('∀', 'x', BinOp('→', phrase_to_pred(m_no.group(1), 'x'), Not(phrase_to_pred(m_no.group(2), 'x'))))
    m_no2 = re.match(r'^no\s+(.*?)\s+(?:is|are|was|were|work|works|have|has|go|goes|can|will)\s+(.*)$', s_low)
    if m_no2:
        return Quant('∀', 'x', BinOp('→', phrase_to_pred(m_no2.group(1), 'x'), Not(phrase_to_pred(m_no2.group(2), 'x'))))

    m_rel = re.match(r'^(?:all\s+|every\s+|any\s+)?(.*?)\s+(?:who|that|which)\s+(.*?)\s+(?:are|is|was|were|will|can|have|has|do|does|chaperone|attend|schedule|work|use|rank)\s+(.*)$', s_low)
    if m_rel:
        ant = phrase_to_pred(m_rel.group(1) + " " + m_rel.group(2), 'x')
        con = phrase_to_pred(m_rel.group(3), 'x')
        return Quant('∀', 'x', BinOp('→', ant, con))

    m_all = re.match(r'^(?:all|every|any|each)\s+(.*?)\s+(?:are|is|was|were|have|has|work|works|use|uses|go|goes|will|can)\s+(.*)$', s_low)
    if m_all:
        return Quant('∀', 'x', BinOp('→', phrase_to_pred(m_all.group(1), 'x'), phrase_to_pred(m_all.group(2), 'x')))

    m_some = re.match(r'^some\s+(.*?)\s+(?:are|is|was|were|have|has|can|go|goes|sting|stings)\s+(.*)$', s_low)
    if m_some:
        return Quant('∃', 'x', BinOp('∧', phrase_to_pred(m_some.group(1), 'x'), phrase_to_pred(m_some.group(2), 'x')))

    for ent in sorted(known_entities, key=len, reverse=True):
        if ent in s_low:
            matched_ent = ent.replace(' ', '_')
            rest = s_low.replace(ent, ' ', 1)
            return phrase_to_pred(rest, matched_ent)

    m_pl = re.match(r'^([a-z\s]+?)\s+(?:are|is)\s+(.*)$', s_low)
    if m_pl:
        return Quant('∀', 'x', BinOp('→', phrase_to_pred(m_pl.group(1), 'x'), phrase_to_pred(m_pl.group(2), 'x')))

    return phrase_to_pred(s_low, 'c0')


def solve_folio_record(r: dict):
    if r.get('premises-FOL') and r.get('conclusion-FOL'):
        p_exprs = [parse_fol_formula(line.strip()) for line in r['premises-FOL'].split('\n') if line.strip()]
        c_expr = parse_fol_formula(r['conclusion-FOL'].strip())
        has_ex = any('∃' in line for line in r['premises-FOL'].split('\n'))
        res = solve_fol_exprs(p_exprs, c_expr, has_existential=has_ex, use_compound_align=False)
        if res != 'Uncertain':
            return res
        return solve_fol_exprs(p_exprs, c_expr, has_existential=has_ex, use_compound_align=True)
    else:
        prem_text = r.get('premises', r.get('context', ''))
        conc_text = r.get('conclusion', r.get('statement', ''))
        ents = set()
        for m in re.finditer(r'\b([A-Z][a-zA-Z0-9\-\.]+(?:\s+[A-Z][a-zA-Z0-9\-\.]+)*)\b', prem_text + "\n" + conc_text):
            w = m.group(1).lower()
            if w not in ('all', 'every', 'any', 'some', 'no', 'if', 'when', 'people', 'everyone', 'someone', 'there', 'either', 'neither'):
                ents.add(w)
        p_exprs = [parse_nl_sentence_to_fol(line.strip(), ents) for line in prem_text.split('\n') if line.strip()]
        c_expr = parse_nl_sentence_to_fol(conc_text.strip(), ents)
        return solve_fol_exprs(p_exprs, c_expr, has_existential=True, use_compound_align=True)


RULETAKER_PATTERNS = [
    r'^Assuming (.*), we can say (.*)\.$',
    r'^If (.*), then (.*)\.$',
    r'^If (.*) then (.*)\.$',
    r'^Based on (.*), it is evident that (.*)\.$',
    r'^Given that (.*), it follows that (.*)\.$',
    r'^When (.*), we can conclude (.*)\.$',
]

def normalize_prop(s: str) -> str:
    return re.sub(r'\s+', ' ', s.strip().rstrip('.').strip().lower())


def parse_nl_atomic_clause(text: str, default_var: str = "?x") -> Literal:
    t = text.strip().rstrip('.').strip()
    t_low = t.lower()
    pronouns = {'someone', 'something', 'anyone', 'anything', 'everyone', 'everything',
                'they', 'it', 'he', 'she', 'a person', 'an individual', 'one'}
    m = re.match(r'^(.*?)\s+(?:is|are|was|were)\s+not\s+(.*)$', t_low)
    if m:
        subj_raw, pred_raw = m.group(1).strip(), m.group(2).strip()
        subj = default_var if subj_raw in pronouns else subj_raw.replace(' ', '_')
        pred = re.sub(r'^(?:a|an|the)\s+', '', pred_raw).replace(' ', '_')
        return Literal(pred, [subj], negated=True)
    m = re.match(r'^(.*?)\s+(?:is|are|was|were)\s+(.*)$', t_low)
    if m:
        subj_raw, pred_raw = m.group(1).strip(), m.group(2).strip()
        subj = default_var if subj_raw in pronouns else subj_raw.replace(' ', '_')
        pred = re.sub(r'^(?:a|an|the)\s+', '', pred_raw).replace(' ', '_')
        return Literal(pred, [subj], negated=False)
    m = re.match(r'^(.*?)\s+(?:does|do|did)\s+not\s+([a-z]+)\s+(.*)$', t_low)
    if m:
        subj_raw, verb, obj_raw = m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
        subj = default_var if subj_raw in pronouns else subj_raw.replace(' ', '_')
        obj = default_var if obj_raw in pronouns else obj_raw.replace(' ', '_')
        return Literal(verb, [subj, obj], negated=True)
    neg = False
    if t_low.startswith('not '):
        neg = True
        t_low = t_low[4:].strip()
    return Literal(normalize_prop(t_low), [], negated=neg)


def parse_ruletaker_or_general_horn(context: str, statement: str):
    if "Facts:" in context and "Rules:" in context:
        parts = context.split("Rules:")
        fact_lines = [l.strip() for l in parts[0].replace("Facts:", "").strip().split("\n") if l.strip()]
        rule_lines = [l.strip() for l in parts[1].strip().split("\n") if l.strip()]
        facts = [Literal(normalize_prop(f), [], False) for f in fact_lines]
        rules = []
        for rl in rule_lines:
            for p in RULETAKER_PATTERNS:
                m = re.match(p, rl.strip(), flags=re.IGNORECASE)
                if m:
                    ant_raw, con_raw = m.group(1), m.group(2)
                    ants = [Literal(normalize_prop(x), [], False) for x in re.split(r'\band\b', ant_raw)]
                    cons = [Literal(normalize_prop(x), [], False) for x in re.split(r'\band\b', con_raw)]
                    for c in cons:
                        rules.append(Rule(ants, c, raw_text=rl))
                    break
        query = Literal(normalize_prop(statement), [], False)
        return facts, rules, query, False

    sentences = [s.strip() + "." for s in re.split(r'\.\s*|\n+', context.strip()) if s.strip()]
    facts = []
    rules = []
    has_quantified = False

    for s in sentences:
        s_clean = s.strip()
        matched_rule = False
        for p in RULETAKER_PATTERNS:
            m = re.match(p, s_clean, flags=re.IGNORECASE)
            if m:
                matched_rule = True
                ant_raw, con_raw = m.group(1).strip(), m.group(2).strip()
                if re.search(r'\b(someone|something|anyone|anything|they|it|he|she)\b', s_clean.lower()):
                    has_quantified = True
                    ant_parts = [x.strip() for x in re.split(r'\band\b', ant_raw) if x.strip()]
                    ants = []
                    for ap in ant_parts:
                        if not re.search(r'\b(is|are|was|were|does|do)\b', ap.lower()):
                            ap = "someone is " + ap
                        ants.append(parse_nl_atomic_clause(ap, "?x"))
                    con_lit = parse_nl_atomic_clause(con_raw, "?x")
                    rules.append(Rule(ants, con_lit, raw_text=s_clean))
                    if len(ants) == 1:
                        rules.append(Rule([con_lit.negate()], ants[0].negate(), raw_text=f"Contrapositive({s_clean})"))
                else:
                    ants = [parse_nl_atomic_clause(x, "?x") for x in re.split(r'\band\b', ant_raw) if x.strip()]
                    cons = [parse_nl_atomic_clause(x, "?x") for x in re.split(r'\band\b', con_raw) if x.strip()]
                    for c in cons:
                        rules.append(Rule(ants, c, raw_text=s_clean))
                break
        if matched_rule:
            continue

        m_all = re.match(r'^(?:All\s+)?([a-zA-Z,\s]+?)\s+(?:things|people|animals|beings|individuals)\s+are\s+(not\s+)?([a-zA-Z\s]+)\.$', s_clean, flags=re.IGNORECASE)
        if m_all:
            has_quantified = True
            adjs = [a.strip().lower() for a in re.split(r',|\band\b', m_all.group(1)) if a.strip()]
            is_neg = bool(m_all.group(2))
            con_pred = m_all.group(3).strip().lower().replace(' ', '_')
            ants = [Literal(a.replace(' ', '_'), ["?x"], False) for a in adjs]
            head = Literal(con_pred, ["?x"], is_neg)
            rules.append(Rule(ants, head, raw_text=s_clean))
            if len(ants) == 1:
                rules.append(Rule([head.negate()], ants[0].negate(), raw_text=f"Contrapositive({s_clean})"))
            continue

        facts.append(parse_nl_atomic_clause(s_clean, "?x"))

    query = parse_nl_atomic_clause(statement, "?x")
    return facts, rules, query, has_quantified


def parse_causal_lit(s: str):
    s = s.strip().rstrip('.')
    if s.lower().startswith('not '):
        return (s[4:].strip(), False)
    return (s, True)


def solve_counterbench_scm(given_info: str, question: str):
    """Pearl's 3-Step Counterfactual Reasoning (Abduction -> Action -> Prediction)."""
    trace = []
    sents = [s.strip() for s in given_info.strip().split('.') if s.strip()]
    cs = sents[0]
    if cs.startswith('We know that '):
        cs = cs[len('We know that '):]

    obs = {}
    for s in sents[1:]:
        if s.startswith('We observed '):
            obs_str = s[len('We observed '):].strip()
            for part in re.split(r'\band\b|,', obs_str):
                if part.strip():
                    v, val = parse_causal_lit(part.strip())
                    obs[v] = val

    raw_parts = [p.strip() for p in cs.split(',') if p.strip()]
    clauses = []
    for p in raw_parts:
        if re.search(r'\b(causes|cause)\b', p):
            if p.startswith('and '):
                p = p[4:].strip()
            clauses.append(p)
        else:
            clauses[-1] = clauses[-1] + ', ' + p

    equations = []
    all_vars = set(obs.keys())
    for cl in clauses:
        m = re.match(r'^(.*?)\s+(?:together\s+)?(?:causes|cause)\s+(.*)$', cl)
        if not m:
            continue
        lhs_str, rhs_str = m.group(1).strip(), m.group(2).strip()
        if ' or ' in lhs_str:
            op = 'OR'
            lhs_lits = [parse_causal_lit(x) for x in re.split(r'\bor\b|,', lhs_str) if x.strip()]
        else:
            op = 'AND'
            lhs_lits = [parse_causal_lit(x) for x in re.split(r'\band\b|,', lhs_str) if x.strip()]
        rhs_lits = [parse_causal_lit(x) for x in re.split(r'\band\b|,', rhs_str) if x.strip()]
        for v, _ in lhs_lits:
            all_vars.add(v)
        for v, pol in rhs_lits:
            all_vars.add(v)
            equations.append((v, pol, op, lhs_lits))
            op_sym = " ∨ " if op == "OR" else " ∧ "
            lhs_fmt = op_sym.join((lv if lpol else f"¬{lv}") for lv, lpol in lhs_lits)
            rhs_fmt = v if pol else f"¬{v}"
            trace.append(f"Structural Eq: {lhs_fmt} ⇒ {rhs_fmt}")

    interventions = {}
    q = question.strip()
    m1 = re.match(r'^Would\s+([A-Za-z0-9_]+)\s+occur\s+if\s+(.*?)\s+instead\s+of\s+(.*?)\?$', q)
    m2 = re.match(r'^Would\s+([A-Za-z0-9_]+)\s+occur\s+if\s+(.*?)\?$', q)
    m3 = re.match(r'^Assume\s+(.*?),\s+and\s+based\s+on\s+this\s+assumption,\s+further\s+suppose\s+(.*?)\.\s+Would\s+([A-Za-z0-9_]+)\s+occur\?$', q)
    if m1:
        target_q = m1.group(1)
        v, val = parse_causal_lit(m1.group(2))
        interventions[v] = val
    elif m2:
        target_q = m2.group(1)
        for part in re.split(r'\band\b|,', m2.group(2)):
            if part.strip():
                v, val = parse_causal_lit(part.strip())
                interventions[v] = val
    elif m3:
        target_q = m3.group(3)
        for grp in [m3.group(1), m3.group(2)]:
            for part in re.split(r'\band\b|,', grp):
                if part.strip():
                    v, val = parse_causal_lit(part.strip())
                    interventions[v] = val
    else:
        target_q = ""

    eq_by_var = {}
    for v, pol, op, lhs_lits in equations:
        eq_by_var.setdefault(v, []).append((pol, op, lhs_lits))

    state = {}
    for k, val in obs.items():
        state[k] = val
        trace.append(f"Abduction (Observation): {k} = {val}")
    for k, val in interventions.items():
        state[k] = val
        trace.append(f"Action [do-Intervention]: do({k} = {val})")

    for _ in range(len(all_vars) + 2):
        for v, eq_list in eq_by_var.items():
            if v in interventions:
                continue
            if v in obs and v not in eq_by_var:
                continue
            vals = []
            for pol, op, lhs_lits in eq_list:
                if op == 'AND':
                    lhs_val = all(state.get(lv, False) == lpol for lv, lpol in lhs_lits)
                else:
                    lhs_val = any(state.get(lv, False) == lpol for lv, lpol in lhs_lits)
                vals.append((pol, lhs_val))
            pol, lhs_val = vals[0]
            state[v] = lhs_val if pol else (not lhs_val)

    ans_bool = bool(state.get(target_q, False))
    trace.append(f"Prediction: {target_q} = {ans_bool} → Answer: {'yes' if ans_bool else 'no'} (Flag: {ans_bool})")
    return ans_bool, ('yes' if ans_bool else 'no'), trace

In [ ]:
# ==============================================================================
# 5. UNIFIED END-TO-END CLASSICAL REASONING AGENT & SUBMISSION GENERATOR
# ==============================================================================

class ClassicalReasoningAgent:
    """
    End-to-End Classical AI Reasoning System.
    Takes a knowledge base (context) and a query (statement) or dataset record
    and produces a logical entailment prediction: True or False.
    """
    def __init__(self, search_strategy: str = "astar"):
        self.search_strategy = search_strategy

    def predict_record(self, record: dict, return_trace: bool = False):
        # 1. CounterBench causal counterfactual query
        if 'given_info' in record and 'question' in record:
            ans_bool, _, trace = solve_counterbench_scm(record['given_info'], record['question'])
            return (ans_bool, trace) if return_trace else ans_bool

        ctx = record.get('context', record.get('premises', ''))
        stmt = record.get('statement', record.get('conclusion', record.get('question', '')))

        if ctx.strip().startswith('We know that ') and ('causes' in ctx or 'cause' in ctx):
            ans_bool, _, trace = solve_counterbench_scm(ctx, stmt)
            return (ans_bool, trace) if return_trace else ans_bool

        # 2. FOLIO First-Order Logic query (either with FOL annotations or complex NL premises)
        if 'premises-FOL' in record or 'premises' in record:
            res3 = solve_folio_record(record)
            is_true = (res3 == 'True')
            trace = [
                f"Parsed FOL Knowledge Base & Conclusion: {stmt}",
                f"Executed Skolemization, Subject-Sorted Herbrand Grounding, Tseitin CNF, and DPLL/Resolution.",
                f"3-Way Logical Status: {res3} → Entailment Prediction: {is_true}"
            ]
            return (is_true, trace) if return_trace else is_true

        # 3. RuleTaker or General Natural Language Horn / Quantified KB
        facts, rules, query, is_fol = parse_ruletaker_or_general_horn(ctx, stmt)
        pred, trace, steps = backward_chaining_search(facts, rules, query, strategy=self.search_strategy)
        if return_trace:
            _, _, fc_trace, _ = forward_chaining_fol(facts, rules, query)
            return pred, fc_trace + ["--- Backward Chaining (" + self.search_strategy.upper() + ") ---"] + trace
        return pred

    def predict(self, context: str, statement: str, return_trace: bool = False):
        return self.predict_record({'context': context, 'statement': statement}, return_trace=return_trace)


# Demonstrate step-by-step reasoning traces across all paradigms
agent = ClassicalReasoningAgent(search_strategy="astar")

print("=" * 80)
print("TRACE 1: Assignment PDF Example (First-Order Unification + Forward/Backward Chaining)")
print("=" * 80)
bob_ctx = """Bob is young.
Bob is kind.
If someone is young, then they are smart.
If someone is smart and kind, then they are successful."""
bob_q = "Bob is successful."
pred, trace = agent.predict(bob_ctx, bob_q, return_trace=True)
print(f"Query: {bob_q} -> Prediction: {pred}")
for line in trace:
    print("  ", line)

TRACE 1: Assignment PDF Example (First-Order Unification + Forward/Backward Chaining)
Query: Bob is successful. -> Prediction: True
   Initial Facts (2): kind(bob), young(bob)
   Step 1 [FC]: young(bob) ⇒ smart(bob) (via (young(?x) → smart(?x)))
   Step 2 [FC]: smart(bob) ∧ kind(bob) ⇒ successful(bob) (via (smart(?x) ∧ kind(?x) → successful(?x)))
   Goal successful(bob) proved via Forward Chaining in 2 steps!
   --- Backward Chaining (ASTAR) ---
   Init Goal: successful(bob)
   Backchain successful(bob) via (smart(?x) ∧ kind(?x) → successful(?x)) → [smart(bob), kind(bob)]
   Backchain smart(bob) via (young(?x) → smart(?x)) → [young(bob)]
   Fact Match: young(bob) = young(bob)
   Fact Match: kind(bob) = kind(bob)
   All subgoals discharged (ASTAR, 4 expansions)!


In [ ]:
# ==============================================================================
# 6. EVALUATION ON PUBLIC DATASETS & GENERATION OF SUBMISSION.CSV
# ==============================================================================

def find_dataset_dir():
    candidates = [
        "./submission",
        ".",
        "..",
        "../data",
        "./data"
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "ruletaker_test.jsonl")):
            return c
    return "."

data_dir = find_dataset_dir()
submission_rows = []

print("=" * 80)
print("EVALUATING CLASSICAL REASONING SYSTEM ON PUBLIC BENCHMARKS")
print("=" * 80)

overall_correct = 0
overall_total = 0
overall_time = 0.0

benchmarks = [
    ("RuleTaker-Train", "ruletaker_train.jsonl", "rt_train"),
    ("RuleTaker-Test", "ruletaker_test.jsonl", "rt_test"),
    ("RuleTaker-OOD-Test", "ruletaker_ood_test.jsonl", "rt_ood"),
    ("CounterBench-Alpha", "counterbench_alpha.jsonl", "cb"),
    ("FOLIO-Validation", "FOLIO_validation.jsonl", "folio_val"),
    ("FOLIO-Train", "FOLIO_train.jsonl", "folio_train"),
]

eval_correct = 0
eval_total = 0
eval_time = 0.0

for b_name, filename, prefix in benchmarks:
    fpath = os.path.join(data_dir, filename)
    if not os.path.exists(fpath):
        continue
    rows = [json.loads(line) for line in open(fpath, 'r', encoding='utf-8') if line.strip()]
    t0 = time.time()
    correct = 0
    for idx, r in enumerate(rows):
        pred_flag = agent.predict_record(r, return_trace=False)
        # Determine ground-truth boolean flag
        if 'flag' in r:
            gold_flag = (str(r['flag']).strip().lower() == 'true')
        elif 'answer' in r:
            gold_flag = (str(r['answer']).strip().lower() in ('yes', 'true', '1'))
        elif 'label' in r:
            gold_flag = (str(r['label']).strip().lower() in ('true', 'entailment'))
        else:
            gold_flag = None

        if gold_flag is not None and pred_flag == gold_flag:
            correct += 1

        # Record ID for submission.csv (include all held-out evaluation sets)
        if b_name in ("RuleTaker-Test", "RuleTaker-OOD-Test", "CounterBench-Alpha", "FOLIO-Validation"):
            raw_id = r.get('id', r.get('question_id', r.get('example_id', idx)))
            rec_id = f"{prefix}_{raw_id}"
            submission_rows.append({"id": rec_id, "label": "True" if pred_flag else "False"})

    dt = time.time() - t0
    acc = 100.0 * correct / len(rows)
    overall_correct += correct
    overall_total += len(rows)
    overall_time += dt
    if b_name in ("RuleTaker-Test", "RuleTaker-OOD-Test", "CounterBench-Alpha", "FOLIO-Validation"):
        eval_correct += correct
        eval_total += len(rows)
        eval_time += dt
    print(f"{b_name:<20} | Queries: {len(rows):>4d} | Accuracy: {correct:>4d}/{len(rows):<4d} ({acc:6.2f}%) | Inference Time: {dt:6.4f}s ({1000*dt/len(rows):6.3f} ms/query)")

print("-" * 80)
print(f"HELD-OUT EVAL SPLITS | Queries: {eval_total:>4d} | Final Accuracy: {eval_correct}/{eval_total} ({100*eval_correct/eval_total:.2f}%) | Total Inference Time: {eval_time:.4f}s ({1000*eval_time/eval_total:.3f} ms/query)")
print(f"ALL PUBLIC DATASETS  | Queries: {overall_total:>4d} | Final Accuracy: {overall_correct}/{overall_total} ({100*overall_correct/overall_total:.2f}%) | Total Inference Time: {overall_time:.4f}s ({1000*overall_time/overall_total:.3f} ms/query)")
print("=" * 80)


def generate_submission_csv(input_jsonl_or_records=None, output_csv_path="submission.csv"):
    """
    Generates submission.csv with columns `id,label` (`True` or `False`).
    If `input_jsonl_or_records` is a path to a private test JSONL file, evaluates every row in that file;
    otherwise writes the held-out public evaluation predictions.
    """
    rows_to_write = []
    if isinstance(input_jsonl_or_records, str) and os.path.exists(input_jsonl_or_records):
        records = [json.loads(line) for line in open(input_jsonl_or_records, 'r', encoding='utf-8') if line.strip()]
        for idx, r in enumerate(records):
            pred_flag = agent.predict_record(r, return_trace=False)
            rec_id = r.get('id', r.get('question_id', r.get('example_id', idx)))
            rows_to_write.append({"id": rec_id, "label": "True" if pred_flag else "False"})
    elif isinstance(input_jsonl_or_records, list):
        rows_to_write = input_jsonl_or_records
    else:
        rows_to_write = submission_rows

    with open(output_csv_path, "w", newline="", encoding="utf-8") as fp:
        writer = csv.DictWriter(fp, fieldnames=["id", "label"])
        writer.writeheader()
        writer.writerows(rows_to_write)
    print(f"Successfully generated {output_csv_path} with {len(rows_to_write)} predictions.")

generate_submission_csv(submission_rows, "submission.csv")

EVALUATING CLASSICAL REASONING SYSTEM ON PUBLIC BENCHMARKS
RuleTaker-Train      | Queries: 1000 | Accuracy:  983/1000 ( 98.30%) | Inference Time: 0.2664s ( 0.266 ms/query)
RuleTaker-Test       | Queries:  250 | Accuracy:  244/250  ( 97.60%) | Inference Time: 0.0668s ( 0.267 ms/query)
RuleTaker-OOD-Test   | Queries:  375 | Accuracy:  373/375  ( 99.47%) | Inference Time: 0.0915s ( 0.244 ms/query)
CounterBench-Alpha   | Queries: 1000 | Accuracy:  960/1000 ( 96.00%) | Inference Time: 0.2131s ( 0.213 ms/query)
FOLIO-Validation     | Queries:  203 | Accuracy:  169/203  ( 83.25%) | Inference Time: 14.2130s (70.015 ms/query)
FOLIO-Train          | Queries: 1001 | Accuracy:  835/1001 ( 83.42%) | Inference Time: 87.5245s (87.437 ms/query)
--------------------------------------------------------------------------------
HELD-OUT EVAL SPLITS | Queries: 1828 | Final Accuracy: 1746/1828 (95.51%) | Total Inference Time: 14.5843s (7.978 ms/query)
ALL PUBLIC DATASETS  | Queries: 3829 | Final Accuracy: 3

In [ ]:
# 1. Test any custom natural-language knowledge base and query
context = """
Bob is young.
Bob is kind.
If someone is young, then they are smart.
If someone is smart and kind, then they are successful.
"""
query = "Bob is successful."

flag, trace = agent.predict(context, query, return_trace=True)
print(f"Query: {query} -> Predicted Flag: {flag}")
for step in trace:
    print(" ", step)

# 2. Test a specific row from RuleTaker (e.g., row 0 of ruletaker_ood_test.jsonl)
row_rt = json.loads(open(os.path.join(data_dir, "ruletaker_ood_test.jsonl")).readlines()[0])
pred_rt, trace_rt = agent.predict_record(row_rt, return_trace=True)
print(f"\n[RuleTaker] Statement: {row_rt['statement']}")
print(f"Predicted: {pred_rt} | Ground Truth: {row_rt['flag']}")
for step in trace_rt:
    print(" ", step)

# 3. Test a CounterBench counterfactual question (e.g., row 5 of counterbench_alpha.jsonl)
row_cb = json.loads(open(os.path.join(data_dir, "counterbench_alpha.jsonl")).readlines()[5])
pred_cb, trace_cb = agent.predict_record(row_cb, return_trace=True)
print(f"\n[CounterBench] Question: {row_cb['question']}")
print(f"Predicted: {pred_cb} ({'yes' if pred_cb else 'no'}) | Ground Truth: {row_cb['answer']}")
for step in trace_cb:
    print(" ", step)

# 4. Test a FOLIO First-Order Logic problem (e.g., row 9 of FOLIO_validation.jsonl)
row_folio = json.loads(open(os.path.join(data_dir, "FOLIO_validation.jsonl")).readlines()[9])
pred_folio, trace_folio = agent.predict_record(row_folio, return_trace=True)
print(f"\n[FOLIO] Conclusion: {row_folio['conclusion']}")
print(f"Predicted: {pred_folio} | Ground Truth: {row_folio['label'] == 'True'} ({row_folio['label']})")
for step in trace_folio:
    print(" ", step)

Query: Bob is successful. -> Predicted Flag: True
  Initial Facts (2): kind(bob), young(bob)
  Step 1 [FC]: young(bob) ⇒ smart(bob) (via (young(?x) → smart(?x)))
  Step 2 [FC]: smart(bob) ∧ kind(bob) ⇒ successful(bob) (via (smart(?x) ∧ kind(?x) → successful(?x)))
  Goal successful(bob) proved via Forward Chaining in 2 steps!
  --- Backward Chaining (ASTAR) ---
  Init Goal: successful(bob)
  Backchain successful(bob) via (smart(?x) ∧ kind(?x) → successful(?x)) → [smart(bob), kind(bob)]
  Backchain smart(bob) via (young(?x) → smart(?x)) → [young(bob)]
  Fact Match: young(bob) = young(bob)
  Fact Match: kind(bob) = kind(bob)
  All subgoals discharged (ASTAR, 4 expansions)!

[RuleTaker] Statement: The teacher plays instruments.
Predicted: True | Ground Truth: True
  Initial Facts (3): sarah is interesting, sarah solves puzzles, the pumpkin solves puzzles
  Step 1 [FC]: sarah solves puzzles ⇒ the teacher plays instruments (via (sarah solves puzzles → the teacher plays instruments))
  Goal t